# 기능 3 — 상태 변화 분석 (Google Colab T4)

**처리 범위**
- 기능 1·2 추론 결과 파싱 및 발신자별 집계
- 오늘 vs 직전 7일 비교, 3대 지표 연산
- 종합 상태(🟢🟡🟠⚪) 판정 + 자연어 해석 문구
- 산출물: `sample_report.json`

이 노트북은 기능 1·2 모델 추론부터 기능 3 리포트 생성까지 **전체 경로**를 시연합니다.
기능 3 로직만 단독으로 돌리려면 `state_change_analysis.py` + `sample_chat_log.json`만으로 `python state_change_analysis.py` 실행하면 됩니다 (GPU·모델 불필요).

## CELL 1 — 환경 설정 (T4 런타임)

런타임 → **T4 GPU** 선택 후 실행. 기능 3 연산은 CPU만으로도 동작하지만, 기능 1·2 추론에 GPU를 씁니다.

In [1]:
# Colab 기본 패키지 (기능 3은 표준 라이브러리만 사용)
import json
import sys
import subprocess
from datetime import date
from pathlib import Path

# 저장소 자동 clone — Run all만으로 기능3 코드(state_change_analysis.py)와
# 입력(sample_chat_log_raw.json)을 확보한다. (이미 있으면 재사용)
REPO_URL = "https://github.com/SARA-MAYO/ChaeOn-AIProgramming.git"
REPO_DIR = Path("/content/ChaeOn-AIProgramming")
if not REPO_DIR.exists():
    print("저장소 clone 중...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("이미 clone된 저장소를 사용합니다.")

# 작업 디렉터리 = 기능3 폴더 (py 파일·입력 json 위치)
WORK_DIR = REPO_DIR / "feature3"
if not WORK_DIR.exists():
    WORK_DIR = Path(".")  # 폴백: 노트북과 같은 폴더에 파일이 있을 때

sys.path.insert(0, str(WORK_DIR))
print("WORK_DIR:", WORK_DIR.resolve())

저장소 clone 중...
WORK_DIR: /content/ChaeOn-AIProgramming/feature3


In [2]:
import os
from google.colab import drive
import torch, joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── 산출물은 기능별 폴더에 저장됨 (기능1·2 노트북이 MyDrive/feature1, MyDrive/feature2 에 저장) ──
DRIVE = '/content/drive/MyDrive'
F1_DIR = os.path.join(DRIVE, 'feature1')   # 기능1 산출물 폴더
F2_DIR = os.path.join(DRIVE, 'feature2')   # 기능2 산출물 폴더

def _has_weights(d):
    return any(os.path.exists(os.path.join(d, w)) for w in ['model.safetensors', 'pytorch_model.bin'])

problems = []
# 모델 폴더: 폴더 존재 + 가중치 파일 존재까지 확인
for path, desc in {
    os.path.join(F1_DIR, 'chaeon_feature1_checkpoint'): '기능1 KcELECTRA 모델  → 기능1 노트북 Run all 필요',
    os.path.join(F2_DIR, 'chaeon_feature2_model'):      '기능2 KoELECTRA 모델  → 기능2 노트북 Run all 필요',
}.items():
    if not os.path.exists(path):
        problems.append(f"  - {path} 폴더 없음            ({desc})")
    elif not _has_weights(path):
        problems.append(f"  - {path} 폴더는 있으나 가중치(model.safetensors) 없음  ({desc})")
# pkl 파일 (기능1)
for path, desc in {
    os.path.join(F1_DIR, 'svm_model.pkl'):  '기능1 SVM 모델          → 기능1 노트북 Run all 필요',
    os.path.join(F1_DIR, 'vectorizer.pkl'): '기능1 TF-IDF 벡터라이저  → 기능1 노트북 Run all 필요',
}.items():
    if not os.path.exists(path):
        problems.append(f"  - {path} 없음            ({desc})")

if problems:
    raise FileNotFoundError(
        "기능3 실행에 필요한 파일이 Google Drive(MyDrive/feature1, MyDrive/feature2)에 없거나 불완전합니다:\n"
        + "\n".join(problems)
        + "\n\n→ 부족한 기능의 노트북을 먼저 'Run all' 하면 해당 기능 폴더에 자동 저장됩니다."
    )
print("필수 모델 파일(가중치 포함) 확인 완료")

# 기능1 로드 (MyDrive/feature1)
f1_ckpt = os.path.join(F1_DIR, 'chaeon_feature1_checkpoint')
f1_model = AutoModelForSequenceClassification.from_pretrained(f1_ckpt).to(device)
f1_tokenizer = AutoTokenizer.from_pretrained(f1_ckpt)
f1_svm = joblib.load(os.path.join(F1_DIR, 'svm_model.pkl'))
f1_vec = joblib.load(os.path.join(F1_DIR, 'vectorizer.pkl'))

# 기능2 로드 (MyDrive/feature2)
f2_ckpt = os.path.join(F2_DIR, 'chaeon_feature2_model')
f2_model = AutoModelForSequenceClassification.from_pretrained(f2_ckpt).to(device)
f2_tokenizer = AutoTokenizer.from_pretrained(f2_ckpt)

print("기능1·2 모델 로드 완료")

Mounted at /content/drive
필수 모델 파일(가중치 포함) 확인 완료


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

기능1·2 모델 로드 완료


## CELL 2 — 기능 1·2 모델 추론 인터페이스

앞 셀에서 **Drive로부터 자동 로드한** 기능 1·2 모델(`f1_*`, `f2_*`)을 호출해 원문 텍스트를 라벨링합니다.
별도 수정 없이 그대로 실행하면 됩니다.

In [3]:
import torch
import torch.nn.functional as F

# 기능1·2 추론 래퍼 — 모델은 CELL3에서 f1_*/f2_* 변수로 미리 로드해 둔다.

def run_feature1_emotion(text: str) -> str:
    # f1_model     : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KcELECTRA 모델 (epoch 2 체크포인트)
    # f1_tokenizer : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KcELECTRA 토크나이저
    # f1_svm       : 새 셀(Cell 1-A)에서 Drive로부터 로드한 SVM 모델 (svm_model.pkl)
    # f1_vec       : 새 셀(Cell 1-A)에서 Drive로부터 로드한 TF-IDF 벡터라이저 (vectorizer.pkl)
    # device       : 새 셀(Cell 1-A)에서 설정한 'cuda' 또는 'cpu'
    f1_model.eval()
    inputs = f1_tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(device)
    with torch.no_grad():
        kc_probs = F.softmax(f1_model(**inputs).logits, dim=-1).cpu().numpy()[0]
    svm_probs = f1_svm.predict_proba(f1_vec.transform([text]))[0]
    prob_pos = (kc_probs[1] * 0.7) + (svm_probs[1] * 0.3)  # KcELECTRA 7 : SVM 3 앙상블
    prob_neg = (kc_probs[0] * 0.7) + (svm_probs[0] * 0.3)
    return "negative" if prob_pos <= prob_neg else "positive"
def run_feature2_aggression(text: str) -> int:
    # f2_model     : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KoELECTRA-small 모델 (checkpoint-5199)
    # f2_tokenizer : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KoELECTRA-small 토크나이저
    f2_model.eval()
    inputs = f2_tokenizer(
        text, return_tensors="pt", padding=True, truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = f2_model(**inputs)
        pred_label = torch.argmax(outputs.logits, dim=-1).item()  # 0=비공격, 1=약한공격, 2=강한공격
    return int(pred_label)
def label_messages(raw_messages: list[dict]) -> list[dict]:
    # run_feature1_emotion, run_feature2_aggression 모두 위 f1_*/f2_* 변수에 의존
    # → 반드시 Cell 1-A(모델 로드 셀) 실행 후에 이 셀을 실행해야 함
    labeled = []
    for msg in raw_messages:
        text = msg["text"]
        labeled.append({
            "message_id":       msg["message_id"],
            "sender_id":        msg["sender_id"],
            "timestamp":        msg["timestamp"],
            "emotion_label":    run_feature1_emotion(text),    # 'negative' 또는 'positive'
            "aggression_label": run_feature2_aggression(text), # 0, 1, 2 정수
        })
    return labeled

## CELL 3 — 입력 데이터 로드

원문 텍스트(`sample_chat_log_raw.json`)가 있으면 기능 1·2 모델로 라벨링하고,
없으면 라벨이 이미 붙은 `sample_chat_log.json`을 그대로 사용합니다.

In [4]:
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

# 입력 우선순위:
#   1) chat_log1_raw.json       — 실제 카카오톡 채팅(전처리·익명화 완료) 원문. 기능 1·2 모델로 라벨 생성.
#   2) sample_chat_log_raw.json — 데모용 원문 text 입력(폴백). 기능 1·2 모델로 라벨을 생성.
#   3) sample_chat_log.json     — 이미 라벨이 붙은 입력. 모델 추론 없이 기능 3만 실행.
INPUT_CANDIDATES = ["chat_log1_raw.json", "sample_chat_log_raw.json", "sample_chat_log.json"]
RAW_CHAT_MESSAGES = []
INPUT_NAME = None
for name in INPUT_CANDIDATES:
    path = WORK_DIR / name
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            RAW_CHAT_MESSAGES = json.load(f)
        INPUT_NAME = name
        print(f"✅ {name} 로드 완료: 총 {len(RAW_CHAT_MESSAGES)}건")
        break
else:
    print("⚠️ 입력 파일이 없습니다. (chat_log1_raw.json 또는 sample_chat_log_raw.json 업로드)")

# ── 이번 실행 산출물 폴더 (입력 로그별·실행시각별로 분리) ──
#   로그를 여러 개 돌려도 서로 덮어쓰지 않도록 outputs/<입력파일명>_<시각>/ 아래에 모은다.
#   (저장소의 sample_chat_log.json 같은 원본 입력은 절대 덮어쓰지 않는다.)
_KST = timezone(timedelta(hours=9))
RUN_STAMP = datetime.now(_KST).strftime("%Y%m%d_%H%M%S")
INPUT_STEM = Path(INPUT_NAME).stem if INPUT_NAME else "run"
RUN_DIR = WORK_DIR / "outputs" / f"{INPUT_STEM}_{RUN_STAMP}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 이번 실행 산출물 폴더: {RUN_DIR}")

# Drive 백업도 같은 폴더 구조로 (마운트돼 있으면). 안 되면 None 으로 두고 조용히 건너뛴다.
DRIVE_RUN_DIR = Path(f"/content/drive/MyDrive/feature3_outputs/{INPUT_STEM}_{RUN_STAMP}")
try:
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    DRIVE_RUN_DIR = None

# 기준일은 state_change_analysis.py 내부에서 데이터 최신 날짜를 기준으로 자동 보정되므로
# 여기서는 None으로 설정합니다. (6일 데이터가 무시되지 않게 함)
REFERENCE_DATE = None

✅ chat_log1_raw.json 로드 완료: 총 1271건
📁 이번 실행 산출물 폴더: /content/ChaeOn-AIProgramming/feature3/outputs/chat_log1_raw_20260608_200447


## CELL 4 — 기능 3 모듈 로드 (`state_change_analysis.py`)

저장소의 py 파일을 import합니다. **첫 셀에서 저장소가 자동 clone**되므로 별도 업로드·경로 수정이 필요 없습니다.

In [5]:
from state_change_analysis import (
    MIN_MSG_COUNT,
    MIN_BASELINE_MSG_COUNT,
    run,
    group_messages_by_sender,
    split_time_window,
)

# 입력 데이터에 이미 라벨이 존재하는지 확인 (시연용 JSON 대응)
if RAW_CHAT_MESSAGES and "emotion_label" in RAW_CHAT_MESSAGES[0]:
    print("💡 입력 데이터에 이미 감정/공격성 라벨이 존재합니다. 모델 추론을 생략합니다.")
    labeled_messages = RAW_CHAT_MESSAGES
elif RAW_CHAT_MESSAGES:
    print("💡 텍스트 원문만 존재합니다. 기능 1·2 모델을 구동하여 라벨링을 시작합니다...")
    labeled_messages = label_messages(RAW_CHAT_MESSAGES)
else:
    labeled_messages = []

if labeled_messages:
    print(f"라벨링 완료: {len(labeled_messages)}건")
    print("샘플 1건:", labeled_messages[0])

💡 텍스트 원문만 존재합니다. 기능 1·2 모델을 구동하여 라벨링을 시작합니다...
라벨링 완료: 1271건
샘플 1건: {'message_id': 1, 'sender_id': 'User_A', 'timestamp': '2025-04-07T16:58:00+09:00', 'emotion_label': 'negative', 'aggression_label': 0}


In [6]:
# (통합 검증) 기능 1·2 모델이 실제로 라벨링한 결과를 '이번 실행 폴더'에 저장.
#   - RUN_DIR/labeled.json = "기능 1·2의 실제 출력 = 기능 3 입력"
#   - 저장소의 sample_chat_log.json은 '덮어쓰지 않는다' → 데모 입력이 보존되고,
#     채팅 로그를 여러 개 돌려도 실행 폴더(outputs/<입력명>_<시각>/)별로 결과가 따로 남는다.
#   - 라벨이 이미 있던 입력(=모델 추론을 안 한 경우)은 저장 대상이 아니다.
import shutil

made_by_model = bool(RAW_CHAT_MESSAGES) and "emotion_label" not in RAW_CHAT_MESSAGES[0]
if made_by_model and labeled_messages:
    out_path = RUN_DIR / "labeled.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(labeled_messages, f, ensure_ascii=False, indent=2)
    if DRIVE_RUN_DIR:
        shutil.copy(str(out_path), str(DRIVE_RUN_DIR / "labeled.json"))
    print(f"✅ 기능 1·2 실제 출력 저장: {out_path} ({len(labeled_messages)}건)")
else:
    print("ℹ️ 라벨이 이미 있던 입력이라 저장을 건너뜁니다. (모델 추론 결과만 저장 대상)")

✅ 기능 1·2 실제 출력 저장: /content/ChaeOn-AIProgramming/feature3/outputs/chat_log1_raw_20260608_200447/labeled.json (1271건)


## CELL 4.7 — 기능 1·2 라벨 검수 (전체는 파일로 · 화면엔 랜덤 20건)

기능 1·2 모델이 카톡 원문에 **실제로 붙인 감정(긍/부정)·공격성(0/1/2) 라벨**을 검수합니다.
카톡 메시지는 양이 많아(수백~천 건) 전부를 화면에 띄우지 않고 이렇게 나눕니다:

- **전체 검수 결과는 파일로** — `label_review.csv`(엑셀용) · `label_review.txt`(읽기용)에 `원문 → 감정 / 공격성`을 전부 저장 (제출물)
- **화면엔 '검사 완료' + 분포 요약 한 줄** — 긍/부정·공격 0/1/2가 각각 몇 건인지
- **화면 미리보기는 랜덤 20건만** — `seed=42`로 고정해 누가 돌려도 같은 20건이 나오게 함(재현성)

> 카톡 실데이터엔 정답 라벨이 없어 **정확도(Accuracy)는 낼 수 없습니다.** 이 셀은 사람이 눈으로 검수하는 용도예요.
> 모델 자체의 정답셋 기준 정량 성능(Accuracy·Macro-F1·혼동행렬)은 기능1 `result.txt`, 기능2 `test_result.txt`에 있습니다.

In [7]:
# ── 기능 1·2 라벨 검수: 전체는 파일로 저장, 화면엔 랜덤 20건 미리보기만 ──
#   카톡 실데이터엔 정답이 없어 정확도(Accuracy)는 못 낸다. 사람이 눈으로 검수하는 용도.
#   전체 메시지를 터미널에 다 쏟으면 양이 많으므로:
#     (1) 전체 (원문 → 라벨) 검수 결과는 RUN_DIR/label_review.csv · label_review.txt 로 저장
#     (2) 화면에는 '검사 완료' + 분포 요약 한 줄 + 재현 가능한(seed=42) 랜덤 20건 미리보기만 출력
import csv
import random
import shutil
from collections import Counter

assert labeled_messages, "라벨링된 메시지가 없습니다. CELL 3·4를 먼저 실행하세요."

# 원문(text)은 라벨링 입력(RAW_CHAT_MESSAGES)에, 라벨은 labeled_messages 에 있다. message_id 로 매칭.
_text_by_id = {m.get("message_id"): m.get("text", "") for m in RAW_CHAT_MESSAGES}
has_text = any(_text_by_id.values())

EMO = {"positive": "긍정", "negative": "부정", "neutral": "중립"}
def _emo(k): return EMO.get(k, str(k))
def _agg(v): return f"공격{v}"
def _txt(mid): return _text_by_id.get(mid, "") or "(원문 없음 — 라벨만 입력됨)"

n = len(labeled_messages)
emo_cnt = Counter(m["emotion_label"] for m in labeled_messages)
agg_cnt = Counter(m["aggression_label"] for m in labeled_messages)

# ===== (1) 전체 검수 결과를 이번 실행 폴더에 저장 (제출물) =====
csv_path = RUN_DIR / "label_review.csv"
txt_path = RUN_DIR / "label_review.txt"

with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:   # utf-8-sig: 엑셀 한글 깨짐 방지
    w = csv.writer(f)
    w.writerow(["message_id", "timestamp", "sender_id", "text", "emotion", "aggression"])
    for m in labeled_messages:
        w.writerow([m["message_id"], m["timestamp"], m["sender_id"],
                    _txt(m["message_id"]), _emo(m["emotion_label"]), m["aggression_label"]])

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"기능 1·2 라벨 검수 결과 (총 {n}건)\n")
    f.write("형식:  [발신자 | 시각] \"원문\"  →  감정 / 공격성\n")
    f.write("=" * 70 + "\n")
    for m in labeled_messages:
        t = str(m["timestamp"])[:16].replace("T", " ")
        f.write(f'[{m["sender_id"]} | {t}] "{_txt(m["message_id"])}"  →  '
                f'{_emo(m["emotion_label"])} / {_agg(m["aggression_label"])}\n')

# Drive 백업 (같은 실행 폴더 구조로) — 마운트 안 됐으면 건너뜀
if DRIVE_RUN_DIR:
    for p in (csv_path, txt_path):
        try:
            shutil.copy(str(p), str(DRIVE_RUN_DIR / p.name))
        except Exception:
            pass

# ===== (2) 화면 출력: '검사 완료' + 분포 요약 한 줄 =====
emo_brief = " / ".join(f"{_emo(k)}{emo_cnt[k]}" for k in ["positive", "negative", "neutral"] if emo_cnt.get(k))
agg_brief = " / ".join(f"{k}:{agg_cnt[k]}" for k in [0, 1, 2] if agg_cnt.get(k))
if not has_text:
    print("⚠️ 입력에 원문(text)이 없어 라벨만 저장했습니다. 원문 검수는 *_raw.json 입력으로 다시 라벨링하세요.")
print(f"✅ 라벨 검수 파일 생성: label_review.csv / label_review.txt  (총 {n}건)")
print(f"   저장 폴더: {RUN_DIR}")
print(f"   감정 {emo_brief}  ·  공격 {agg_brief}")

# ===== (3) 재현 가능한 랜덤 20건 미리보기 (seed=42 고정) =====
random.seed(42)   # 프로젝트 재현성 기준(Seed=42) — 누가 돌려도 같은 20건이 나오게 고정
k = min(20, n)
sample = sorted(random.sample(labeled_messages, k), key=lambda m: m["message_id"])
print(f"\n── 라벨 검수 미리보기 (랜덤 {k}건, seed=42) ──")
for m in sample:
    t = str(m["timestamp"])[:16].replace("T", " ")
    print(f'[{m["sender_id"]} | {t}] "{_txt(m["message_id"])}"  →  '
          f'{_emo(m["emotion_label"])} / {_agg(m["aggression_label"])}')
print(f"\n전체 {n}건은 {csv_path} 참고")

✅ 라벨 검수 파일 생성: label_review.csv / label_review.txt  (총 1271건)
   저장 폴더: /content/ChaeOn-AIProgramming/feature3/outputs/chat_log1_raw_20260608_200447
   감정 긍정905 / 부정366  ·  공격 0:1207 / 1:62 / 2:2

── 라벨 검수 미리보기 (랜덤 20건, seed=42) ──
[User_A | 2025-04-14 13:10] "사진 한번 찍어주실수 있나요..?"  →  긍정 / 공격0
[User_C | 2025-04-14 13:11] "30분에 꺼낼때찍을게요"  →  긍정 / 공격0
[User_C | 2025-04-14 13:25] "총 7개있네요"  →  긍정 / 공격0
[User_A | 2025-04-14 13:38] "네 !"  →  긍정 / 공격0
[User_A | 2025-05-13 10:37] "맞는거 같아요"  →  긍정 / 공격0
[User_C | 2025-05-16 14:48] "제가 조건별로 다 모아놨는데 한분씩 맡으셔서"  →  긍정 / 공격0
[User_C | 2025-05-17 13:10] "아아 맥북이시면 제가 저건할게여"  →  긍정 / 공격1
[User_A | 2025-05-17 13:12] "다시.."  →  부정 / 공격0
[User_B | 2025-05-18 13:18] "이모티콘 이렇게 하는게 맞을까요....?"  →  부정 / 공격0
[User_A | 2025-05-20 18:55] "띠봉"  →  부정 / 공격1
[User_B | 2025-05-20 18:55] "내가 한번 해볼게"  →  긍정 / 공격0
[User_A | 2025-05-20 18:57] "이거"  →  긍정 / 공격0
[User_B | 2025-05-20 19:09] "그냥 내가 보낼까?"  →  긍정 / 공격0
[User_C | 2025-05-20 21:53] "그냥"  →  부정 / 공격0
[User_C | 202

## CELL 5.5 — 날짜별 상태 변화 분석 → `daily_report.json`

각 날짜를 '오늘'로 두고 직전 7일과 비교하여 발신자별 상태 변화를 **날짜별로** 산출합니다.
상세 결과(3대 지표 + 자연어 멘트)는 화면에 출력하지 않고 `daily_report.json`에 저장합니다.

In [8]:
# 날짜별 상태 변화 분석 (각 날짜를 '오늘'로 두고 직전 7일과 비교)
#   분석 로직은 state_change_analysis.run_daily() 단일 구현을 재사용한다. (중복 제거)
#   결과는 RUN_DIR/daily_report.json 으로 저장 (실행 폴더별 분리 → 로그 여러 개 돌려도 안 섞임)
import json
from state_change_analysis import run_daily

daily_reports = run_daily(labeled_messages)

# 상세 리포트는 화면에 출력하지 않고 JSON 파일로만 저장 (+ Drive 백업)
out = RUN_DIR / "daily_report.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(daily_reports, f, ensure_ascii=False, indent=2)

if DRIVE_RUN_DIR:
    import shutil
    shutil.copy(str(out), str(DRIVE_RUN_DIR / "daily_report.json"))

print(f"✅ 리포트 생성 완료. (날짜별 {len(daily_reports)}건)")
print(f"   저장 완료: {out}")
if DRIVE_RUN_DIR:
    print(f"   Drive 백업 완료: {DRIVE_RUN_DIR / 'daily_report.json'}")

✅ 리포트 생성 완료. (날짜별 81건)
   저장 완료: /content/ChaeOn-AIProgramming/feature3/outputs/chat_log1_raw_20260608_200447/daily_report.json
   Drive 백업 완료: /content/drive/MyDrive/feature3_outputs/chat_log1_raw_20260608_200447/daily_report.json


## CELL 5.6 — 시연용 시각화 (전체 요약 표 + 개인별 상세)

`daily_report.json` 내용을 보기 좋게 화면에 띄웁니다.
- **전체 요약**: 날짜 × 사용자 상태를 색칠된 표(🟢🟡🟠⚪)로 한눈에
- **개인별 상세**: 사용자별로 날짜마다 3대 지표 변화 + 자연어 멘트

In [9]:
# ── 시연용 시각화: 전체 요약 표 + 개인별 상세 ──
import json
import pandas as pd
from IPython.display import display, Markdown

# 기본값(False): 개인별 상세에 '분석 가능한 날'만 표시합니다.
# 데이터 부족(⚪)한 날까지 전체를 보고 싶으면 True 로 바꾸세요.
SHOW_ALL_DAYS = False

# 기본값(True): 전체 요약 표에서 '모든 사용자가 ⚪(데이터 부족) 또는 ·(메시지 없음)인 날'은 숨깁니다.
#   (= 넷 다 ⚪, 넷 다 ·, 또는 ⚪+· 섞인 날 → 표에서 제외)
# 모든 날짜를 표에 다 보고 싶으면 False 로 바꾸세요.
HIDE_EMPTY_ROWS = True

# daily_reports 가 메모리에 없으면 이번 실행 폴더의 파일에서 로드
try:
    daily_reports
except NameError:
    with open(RUN_DIR / "daily_report.json", encoding="utf-8") as f:
        daily_reports = json.load(f)

def _emoji(state):  # "🟢 평소와 비슷" → "🟢"
    return state.split()[0] if state else "⚪"

senders = sorted({r["sender_id"] for r in daily_reports})
dates   = sorted({r["date"] for r in daily_reports})

# ===== 1) 전체 요약 — 날짜 × 사용자 상태 표 =====
display(Markdown("# ===== 전체 요약 — 날짜별 사용자 상태 ====="))

grid = {(r["sender_id"], r["date"]): _emoji(r["overall"]["state"]) for r in daily_reports}

# 모든 사용자가 ⚪/·(빈 표시)인 날은 숨김 — 한 명이라도 🟢🟡🟠면 그 날은 노출 (옵션)
EMPTY_MARKS = {"⚪", "·"}
if HIDE_EMPTY_ROWS:
    table_dates = [d for d in dates
                   if not all(grid.get((s, d), "·") in EMPTY_MARKS for s in senders)]
else:
    table_dates = dates

df = pd.DataFrame(
    [[grid.get((s, d), "·") for s in senders] for d in table_dates],
    index=table_dates, columns=senders,
)
df.index.name = "날짜"

COLOR = {"🟢": "#d7f5dd", "🟡": "#fff3cd", "🟠": "#ffd9b3", "⚪": "#eeeeee", "·": "#ffffff"}
def _cell_style(v):
    return f"background-color:{COLOR.get(v, '#ffffff')}; text-align:center; font-size:18px;"

styler = df.style
try:
    styler = styler.map(_cell_style)          # pandas ≥ 2.1
except AttributeError:
    styler = styler.applymap(_cell_style)     # 구버전 호환
display(styler)
display(Markdown("🟢 평소와 비슷 · 🟡 조금 다름 · 🟠 꽤 다름 · ⚪ 데이터 부족 · `·` 메시지 없음"))

# ===== 2) 개인별 상세 — 날짜별 지표 변화 + 멘트 =====
display(Markdown("# ===== 개인별 상세 ====="))
for s in senders:
    rows = [r for r in daily_reports if r["sender_id"] == s]
    if not SHOW_ALL_DAYS:
        rows = [r for r in rows if r["data_sufficient"]]
    display(Markdown(f"## 🧑 {s}"))
    if not rows:
        print("   (분석 가능한 날이 없습니다 — 메시지 10건 이상인 날 필요)")
        continue
    for r in rows:
        print(f"\n▸ {r['date']}   {r['overall']['state']}")
        if r["data_sufficient"]:
            m = r["metrics"]
            print(f"    부정 {m['negative']['change']:+.1f}%p({m['negative']['state']})   "
                  f"공격 {m['aggressive']['change']:+.1f}%p({m['aggressive']['state']})   "
                  f"참여 {m['participation']['change']:+.1f}%({m['participation']['state']})")
        print(f"    💬 {r['overall']['text_interpretation']}")

# ===== 전체 요약 — 날짜별 사용자 상태 =====

,User_A,User_B,User_C,User_D
날짜,,,,
2025-04-14,🟡,⚪,🟢,⚪
2025-05-12,⚪,⚪,🟢,⚪
2025-05-13,🟡,·,⚪,·
2025-05-17,🟢,⚪,🟠,⚪
2025-05-19,🟢,🟡,🟡,⚪
2025-05-20,🟡,🟡,🟡,⚪
2025-05-21,🟢,🟢,🟡,🟡
2025-05-22,🟢,🟢,🟢,🟡


🟢 평소와 비슷 · 🟡 조금 다름 · 🟠 꽤 다름 · ⚪ 데이터 부족 · `·` 메시지 없음

# ===== 개인별 상세 =====

## 🧑 User_A

...


▸ 2025-05-19   🟡 평소와 조금 다름
    부정 +6.9%p(평소보다 높음)   공격 +0.0%p(평소와 비슷)   참여 +1115.0%(평소보다 많음)
    💬 평소보다 대화 참여량이 급격히 늘어난 가운데, 부정적인 감정 표현이 함께 증가한 것이 관측되었습니다. 과도한 열정이나 피로로 인해 날카로워진 것은 아닌지 조원들과 잠시 숨을 골라보세요.

▸ 2025-05-20   🟡 평소와 조금 다름
    부정 +10.0%p(평소보다 높음)   공격 +0.0%p(평소와 비슷)   참여 +1115.5%(평소보다 많음)
    💬 평소보다 대화 참여량이 급격히 늘어난 가운데, 부정적인 감정 표현이 함께 증가한 것이 관측되었습니다. 과도한 열정이나 피로로 인해 날카로워진 것은 아닌지 조원들과 잠시 숨을 골라보세요.

▸ 2025-05-21   🟢 평소와 비슷
    부정 -26.8%p(평소보다 낮음)   공격 +0.0%p(평소와 비슷)   참여 +14.4%(평소와 비슷)
    💬 평소의 안정적인 소통 기저선을 잘 유지하고 있습니다. 지금처럼 건강하고 건설적인 커뮤니케이션을 이어가세요!

▸ 2025-05-22   🟢 평소와 비슷
    부정 -0.6%p(평소와 비슷)   공격 +0.0%p(평소와 비슷)   참여 -18.3%(평소와 비슷)
    💬 평소의 안정적인 소통 기저선을 잘 유지하고 있습니다. 지금처럼 건강하고 건설적인 커뮤니케이션을 이어가세요!


## 🧑 User_C


▸ 2025-04-14   🟢 평소와 비슷
    부정 -2.3%p(평소와 비슷)   공격 +0.0%p(평소와 비슷)   참여 +809.1%(평소보다 많음)
    💬 팀 프로젝트에 적극적으로 참여하며 소통 참여량을 활발히 끌어올리고 계시네요! 조원들에게 든든한 에너지가 되고 있습니다.

▸ 2025-05-12   🟢 평소와 비슷
    부정 -5.0%p(평소보다 낮음)   공격 +0.0%p(평소와 비슷)   참여 +336.7%(평소보다 많음)
    💬 팀 프로젝트에 적극적으로 참여하며 소통 참여량을 활발히 끌어올리고 계시네요! 조원들에게 든든한 에너지가 되고 있습니다.

▸ 2025-05-17   🟠 평소와 꽤 다름
    부정 +9.4%p(평소보다 높음)   공격 +20.5%p(평소보다 높음)   참여 +754.9%(평소보다 많음)
    💬 조원들과의 대화에서 부정적인 감정과 공격적인 표현이 동시에 증가했습니다. 감정이 앞서 오해가 생기지 않도록 잠시 대화를 멈추고 환기해 보는 건 어떨까요?

▸ 2025-05-19   🟡 평소와 조금 다름
    부정 +6.8%p(평소보다 높음)   공격 -6.8%p(평소보다 낮음)   참여 +366.6%(평소보다 많음)
    💬 평소보다 대화 참여량이 급격히 늘어난 가운데, 부정적인 감정 표현이 함께 증가한 것이 관측되었습니다. 과도한 열정이나 피로로 인해 날카로워진 것은 아닌지 조원들과 잠시 숨을 골라보세요.

▸ 2025-05-20   🟡 평소와 조금 다름
    부정 +6.3%p(평소보다 높음)   공격 -7.5%p(평소보다 낮음)   참여 +669.8%(평소보다 많음)
    💬 평소보다 대화 참여량이 급격히 늘어난 가운데, 부정적인 감정 표현이 함께 증가한 것이 관측되었습니다. 과도한 열정이나 피로로 인해 날카로워진 것은 아닌지 조원들과 잠시 숨을 골라보세요.

▸ 2025-05-21   🟡 평소와 조금 다름
    부정 +8.8%p(평소보다 높음)   공격 +3.4%p(평소와 비슷)   참여 +267.

## 🧑 User_D


▸ 2025-05-21   🟡 평소와 조금 다름
    부정 -14.2%p(평소보다 낮음)   공격 +7.8%p(평소보다 높음)   참여 +174.6%(평소보다 많음)
    💬 평소 대비 날카롭거나 공격적인 표현이 관측되었습니다. 메시지를 전송하기 전, 상대방의 입장에서 한 번만 더 읽어보는 건 어떨까요?

▸ 2025-05-22   🟡 평소와 조금 다름
    부정 +19.8%p(평소보다 높음)   공격 -3.4%p(평소와 비슷)   참여 +65.5%(평소보다 많음)
    💬 평소보다 대화 참여량이 급격히 늘어난 가운데, 부정적인 감정 표현이 함께 증가한 것이 관측되었습니다. 과도한 열정이나 피로로 인해 날카로워진 것은 아닌지 조원들과 잠시 숨을 골라보세요.


## CELL 6 — 자체 점검 (sanity check)

`split_time_window` 분리 건수와 가드레일 상수(`MIN_MSG_COUNT` 등)가 의도대로 동작하는지 확인합니다.
결과를 새로 만들지 않고, 분석이 정당하게 수행됐는지 검증하는 용도입니다.

In [10]:
# 자체 점검 — 발신자별 today/baseline 분리 건수 + 가드레일 상수 확인 (재현성·적법성 증빙)
from state_change_analysis import group_messages_by_sender

grouped = group_messages_by_sender(labeled_messages)
for sender_id, msgs in grouped.items():
    today, baseline = split_time_window(msgs, REFERENCE_DATE)
    print(f"[{sender_id}] today={len(today)}, baseline={len(baseline)}")

# 가드레일 상수 확인 (임계값이 의도대로 유지되는지)
assert MIN_MSG_COUNT == 10
assert MIN_BASELINE_MSG_COUNT == 10
print("\n가드레일 상수 OK")

[User_A] today=2, baseline=381
[User_C] today=1, baseline=334
[User_B] today=1, baseline=123
[User_D] today=1, baseline=132

가드레일 상수 OK
